In [1]:
!fuser -k 8000/tcp

In [2]:
# 1. Cài đặt thư viện chạy Server + Cloudflared
!pip install -U transformers accelerate fastapi uvicorn pydantic nest-asyncio -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 84.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.8/69.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 100.6 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gradi

In [3]:
# 2. Load Model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN"))
model_id = "thanhhoangnvbg/empathAI-llama3.1-8b"

print("--- ĐANG TRIỆU HỒI EMPATHAI TỪ CLOUD (DUAL-GPU) ---")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

--- ĐANG TRIỆU HỒI EMPATHAI TỪ CLOUD (DUAL-GPU) ---


config.json:   0%|          | 0.00/955 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [4]:
# 3. Khởi tạo FastAPI Server
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import nest_asyncio
import threading

app = FastAPI()

class ChatRequest(BaseModel):
    messages: list
    max_new_tokens: int = 512
    temperature: float = 0.7

@app.post("/api/chat")
async def chat(request: ChatRequest):
    text = tokenizer.apply_chat_template(
        request.messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=request.max_new_tokens,
            temperature=request.temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    response_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return {"reply": response_text}

# Áp dụng nest_asyncio để cho phép uvicorn chạy ngay trong Jupyter Notebook
nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("Server Cục Bộ đang chạy tại: http://localhost:8000")

Server Cục Bộ đang chạy tại: http://localhost:8000


In [5]:
# 4. Kích hoạt Cloudflare Tunnel (Lấy External Link)
import subprocess
import time
import re

print("--- ĐANG MỞ CLOUDFLARE TUNNEL ---")
process = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--logfile', 'cloudflared.log'], 
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

time.sleep(5) # Đợi tunnel khởi động
with open('cloudflared.log', 'r', encoding='utf-8') as f:
    log_content = f.read()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
    if match:
        url = match.group(0)
        print(f"\n> ✅ TRẠNG THÁI: THÀNH CÔNG")
        print(f"> 🔗 ĐỊA CHỈ API CỦA BẠN LÀ: {url}")
        print(f"> 💡 Copy link trên dán vào Backend (env KAGGLE_INFERENCE_URL)\n")
    else:
        print("Khởi động tunnel chậm, chạy lại cell này xem file log nhé!")


INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


--- ĐANG MỞ CLOUDFLARE TUNNEL ---

> ✅ TRẠNG THÁI: THÀNH CÔNG
> 🔗 ĐỊA CHỈ API CỦA BẠN LÀ: https://stylish-mozilla-washer-guys.trycloudflare.com
> 💡 Copy link trên dán vào Backend (env KAGGLE_INFERENCE_URL)

INFO:     14.187.162.181:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     14.187.162.181:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     14.187.162.181:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     14.187.162.181:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     14.187.162.181:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     14.187.162.181:0 - "GET /chat HTTP/1.1" 404 Not Found
INFO:     14.187.162.181:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     14.187.162.181:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     14.187.162.181:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     14.187.162.181:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     14.187.162.181:0 - "GET / HTTP/1.1" 404 Not Found
